# DS200 - Lab 2


Khởi tạo Spark Session


In [8]:
from pyspark.sql import SparkSession
from datetime import datetime
from pyspark.sql.functions import col, avg, count, explode, split, when, round as spark_round
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, FloatType, LongType

spark = SparkSession.builder \
    .appName("Lab 2") \
    .master("local[*]") \
    .getOrCreate()
    
sc = spark.sparkContext

Load Data


In [ ]:
# Load movies data
movies_rdd = sc.textFile(r'E:\Năm 3\Công nghệ dữ liệu lớn/movies.txt') \
    .map(lambda line: line.split(", ")) \
    .map(lambda fields: (int(fields[0]), fields[1], fields[2]))  # (MovieID, Title, Genres)

# Load ratings_1.txt
ratings_1_rdd = sc.textFile(r'E:\Năm 3\Công nghệ dữ liệu lớn/ratings_1.txt') \
    .map(lambda line: line.split(", ")) \
    .map(lambda fields: (int(fields[0]), int(fields[1]), float(fields[2]), fields[3]))

# Load ratings_2.txt
ratings_2_rdd = sc.textFile(r'E:\Năm 3\Công nghệ dữ liệu lớn/ratings_2.txt') \
    .map(lambda line: line.split(", ")) \
    .map(lambda fields: (int(fields[0]), int(fields[1]), float(fields[2]), fields[3]))

# Union both rating files
all_ratings_rdd = ratings_1_rdd.union(ratings_2_rdd)

# Load users data
users_rdd = sc.textFile(r'E:\Năm 3\Công nghệ dữ liệu lớn/users.txt') \
    .map(lambda line: line.split(", ")) \
    .map(lambda fields: (int(fields[0]), fields[1], int(fields[2]), int(fields[3]), fields[4]))  # (UserID, Gender, Age, Occupation, ZipCode)

# Load occupation data
occupation_rdd = sc.textFile(r'E:\Năm 3\Công nghệ dữ liệu lớn/occupation.txt') \
    .map(lambda line: line.split(", ")) \
    .map(lambda fields: (int(fields[0]), fields[1]))  # (OccupationID, OccupationName)
    
print("Data loaded successfully!")
print(f"Movies: {movies_rdd.count()}")
print(f"Total Ratings: {all_ratings_rdd.count()}")
print(f"Users: {users_rdd.count()}")
print(f"Occupations: {occupation_rdd.count()}")

Bài 1: Tính Điểm Đánh Giá Trung Bình và Tổng Số Lượt Đánh Giá Cho Mỗi Phim

Mục tiêu:

    Tính điểm trung bình cho từng phim từ cả 2 file ratings (ratings_1.txt và ratings_2.txt).
    Tính tổng số lượt đánh giá cho mỗi phim.
    Output: MovieTitle AverageRating: xx (TotalRatings: xx)


In [ ]:
print("\n" + "="*80)
print("BÀI 1: TÍNH ĐIỂM ĐÁNH GIÁ TRUNG BÌNH VÀ TỔNG SỐ LƯỢT ĐÁNH GIÁ CHO MỖI PHIM")
print("="*80)

# Map ratings to (MovieID, (Rating, 1))
ratings_mapped = all_ratings_rdd.map(lambda x: (x[1], (x[2], 1)))

# Reduce to get sum and count for each movie
ratings_aggregated = ratings_mapped.reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))

# Calculate average: (MovieID, (AvgRating, TotalRatings))
ratings_with_avg = ratings_aggregated.map(lambda x: (x[0], (round(x[1][0] / x[1][1], 1), x[1][1])))

# Create movies lookup: (MovieID, Title)
movies_lookup = movies_rdd.map(lambda x: (x[0], x[1]))

# Join with movies to get titles
movies_with_ratings = movies_lookup.join(ratings_with_avg) \
    .map(lambda x: (x[1][0], x[1][1][0], x[1][1][1]))  # (Title, AvgRating, TotalRatings)

# Sort by title
movies_with_ratings_sorted = movies_with_ratings.sortBy(lambda x: x[0])

# Display results
print("\nMovies with Ratings (showing first 20):")
for movie in movies_with_ratings.take(20):
    print(f"{movie[0]} - AverageRating: {movie[1]} (TotalRatings: {movie[2]})")

# Find highest rated movie with at least 5 ratings
movies_min_5 = movies_with_ratings.filter(lambda x: x[2] >= 5)
highest_rated = movies_min_5.sortBy(lambda x: x[1], ascending=False).first()

print("\n" + "="*80)
print(f"{highest_rated[0]} is the highest rated movie with an average rating of {highest_rated[1]} among movies with at least 5 ratings.")
print("="*80)


BÀI 1: TÍNH ĐIỂM ĐÁNH GIÁ TRUNG BÌNH VÀ TỔNG SỐ LƯỢT ĐÁNH GIÁ CHO MỖI PHIM

Movies with Ratings (showing first 20):


Py4JJavaError: An error occurred while calling z:org.apache.spark.api.python.PythonRDD.runJob.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 1 in stage 4.0 failed 1 times, most recent failure: Lost task 1.0 in stage 4.0 (TID 7) (LAPTOP-499558CH executor driver): org.apache.spark.SparkException: Python worker failed to connect back.
	at org.apache.spark.api.python.PythonWorkerFactory.createSimpleWorker(PythonWorkerFactory.scala:252)
	at org.apache.spark.api.python.PythonWorkerFactory.create(PythonWorkerFactory.scala:143)
	at org.apache.spark.SparkEnv.createPythonWorker(SparkEnv.scala:158)
	at org.apache.spark.SparkEnv.createPythonWorker(SparkEnv.scala:178)
	at org.apache.spark.api.python.BasePythonRunner.compute(PythonRunner.scala:261)
	at org.apache.spark.api.python.PythonRDD.compute(PythonRDD.scala:70)
	at org.apache.spark.rdd.RDD.computeOrReadCheckpoint(RDD.scala:374)
	at org.apache.spark.rdd.RDD.iterator(RDD.scala:338)
	at org.apache.spark.rdd.Un

Bài 2: Phân Tích Đánh Giá Theo Thể Loại

Mục tiêu:

        Vì một phim có thể thuộc nhiều thể loại (Genres được phân tách bằng dấu “|”), mapper cần tách riêng từng thể loại của phim đó.
        Tính điểm trung bình (và tổng số lượt đánh giá nếu cần) cho từng thể loại, dựa trên tất cả các phim thuộc thể loại đó.

Output: Genre: AverageRating (TotalRatings)


In [ ]:
print("\n" + "="*80)
print("BÀI 2: PHÂN TÍCH ĐÁNH GIÁ THEO THỂ LOẠI")
print("="*80)

# Create (MovieID, Genres) mapping
movie_genres = movies_rdd.map(lambda x: (x[0], x[2]))

# Join ratings with genres
ratings_with_genres = all_ratings_rdd.map(lambda x: (x[1], x[2])) \
    .join(movie_genres)  # (MovieID, (Rating, Genres))

# Explode genres: (Genre, Rating)
genre_ratings = ratings_with_genres.flatMap(
    lambda x: [(genre, x[1][0]) for genre in x[1][1].split("|")]
)

# Map to (Genre, (Rating, 1))
genre_mapped = genre_ratings.map(lambda x: (x[0], (x[1], 1)))

# Reduce by genre
genre_aggregated = genre_mapped.reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))

# Calculate average for each genre
genre_stats = genre_aggregated.map(
    lambda x: (x[0], round(x[1][0] / x[1][1], 2), x[1][1])
).sortBy(lambda x: x[0])

print("\nGenre Ratings:")
for genre in genre_stats.collect():
    print(f"{genre[0]} - AverageRating: {genre[1]} (TotalRatings: {genre[2]})")

Bài 3: Phân Tích Đánh Giá Theo Giới Tính

Mục tiêu:

    Thực hiện join dữ liệu giữa ratings và users (dựa trên UserID) để lấy thông tin giới tính của người đánh giá.
    Với mỗi phim, tính riêng điểm trung bình từ người dùng nam và nữ.

Output: MovieTitle: Male_Avg, Female_Avg


In [ ]:
print("\n" + "="*80)
print("BÀI 3: PHÂN TÍCH ĐÁNH GIÁ THEO GIỚI TÍNH")
print("="*80)

# Create (UserID, Gender) mapping
user_gender = users_rdd.map(lambda x: (x[0], x[1]))

# Join ratings with gender
ratings_with_gender = all_ratings_rdd.map(lambda x: (x[0], (x[1], x[2]))) \
    .join(user_gender)  # (UserID, ((MovieID, Rating), Gender))

# Map to (MovieID, (Gender, Rating))
movie_gender_ratings = ratings_with_gender.map(
    lambda x: (x[1][0][0], (x[1][1], x[1][0][1]))
)

# Map to ((MovieID, Gender), (Rating, 1))
gender_grouped = movie_gender_ratings.map(
    lambda x: ((x[0], x[1][0]), (x[1][1], 1))
)

# Reduce by (MovieID, Gender)
gender_aggregated = gender_grouped.reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))

# Calculate average: ((MovieID, Gender), AvgRating)
gender_avg = gender_aggregated.map(
    lambda x: (x[0][0], (x[0][1], round(x[1][0] / x[1][1], 2)))
)

# Group by MovieID to get both M and F ratings
gender_by_movie = gender_avg.groupByKey().mapValues(lambda x: dict(x))

# Join with movie titles
movies_gender_ratings = movies_lookup.join(gender_by_movie) \
    .map(lambda x: (x[1][0], x[1][1].get('M', 'N/A'), x[1][1].get('F', 'N/A'))) \
    .sortBy(lambda x: x[0])

print("\nMovies with Gender-specific Ratings (showing first 20):")
for movie in movies_gender_ratings.take(20):
    print(f"{movie[0]} - Male_Avg: {movie[1]}, Female_Avg: {movie[2]}")

Bài 4 (Tùy Chọn): Phân Tích Đánh Giá Theo Nhóm Tuổi
Mục tiêu:

    Phân nhóm người dùng theo độ tuổi (ví dụ: 0-18, 18-35, 35-50, 50+).
    Với mỗi phim, tính điểm trung bình cho mỗi nhóm tuổi.

Output: MovieTitle: [0-18: AvgRating, 18-35: AvgRating, 35-50: AvgRating, 50+: AvgRating]


In [ ]:
print("\n" + "="*80)
print("BÀI 4: PHÂN TÍCH ĐÁNH GIÁ THEO NHÓM TUỔI")
print("="*80)

# Function to determine age group
def get_age_group(age):
    if age < 18:
        return "0-18"
    elif age < 35:
        return "18-35"
    elif age < 50:
        return "35-50"
    else:
        return "50+"

# Create (UserID, AgeGroup) mapping
user_age_group = users_rdd.map(lambda x: (x[0], get_age_group(x[2])))

# Join ratings with age group
ratings_with_age = all_ratings_rdd.map(lambda x: (x[0], (x[1], x[2]))) \
    .join(user_age_group)  # (UserID, ((MovieID, Rating), AgeGroup))

# Map to (MovieID, (AgeGroup, Rating))
movie_age_ratings = ratings_with_age.map(
    lambda x: (x[1][0][0], (x[1][1], x[1][0][1]))
)

# Map to ((MovieID, AgeGroup), (Rating, 1))
age_grouped = movie_age_ratings.map(
    lambda x: ((x[0], x[1][0]), (x[1][1], 1))
)

# Reduce by (MovieID, AgeGroup)
age_aggregated = age_grouped.reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))

# Calculate average
age_avg = age_aggregated.map(
    lambda x: (x[0][0], (x[0][1], round(x[1][0] / x[1][1], 2)))
)

# Group by MovieID
age_by_movie = age_avg.groupByKey().mapValues(lambda x: dict(x))

# Join with movie titles
movies_age_ratings = movies_lookup.join(age_by_movie) \
    .map(lambda x: (
        x[1][0], 
        x[1][1].get('0-18', 'N/A'),
        x[1][1].get('18-35', 'N/A'),
        x[1][1].get('35-50', 'N/A'),
        x[1][1].get('50+', 'N/A')
    )) \
    .sortBy(lambda x: x[0])

print("\nMovies with Age Group Ratings (showing first 20):")
for movie in movies_age_ratings.take(20):
    print(f"{movie[0]} - [0-18: {movie[1]}, 18-35: {movie[2]}, 35-50: {movie[3]}, 50+: {movie[4]}]")

Bài 5: Phân Tích Đánh Giá Theo Occupation (Nghề nghiệp) Của Người Dùng

Mục tiêu:
Tính trung bình rating và tổng số lượt đánh giá cho từng Occupation.

Output: Occupation - TotalRatings: xx, AverageRating: x


In [ ]:
print("\n" + "="*80)
print("BÀI 5: PHÂN TÍCH ĐÁNH GIÁ THEO OCCUPATION")
print("="*80)

# Create (UserID, Occupation) mapping
user_occupation = users_rdd.map(lambda x: (x[0], x[3]))

# Join ratings with occupation
ratings_with_occupation = all_ratings_rdd.map(lambda x: (x[0], x[2])) \
    .join(user_occupation)  # (UserID, (Rating, Occupation))

# Map to (Occupation, (Rating, 1))
occupation_ratings = ratings_with_occupation.map(
    lambda x: (x[1][1], (x[1][0], 1))
)

# Reduce by occupation
occupation_aggregated = occupation_ratings.reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))

# Calculate average and format output
occupation_stats = occupation_aggregated.map(
    lambda x: (x[0], x[1][1], round(x[1][0] / x[1][1], 2))
).sortBy(lambda x: x[0])

print("\nOccupation Ratings:")
for occ in occupation_stats.collect():
    print(f"Occupation {occ[0]} - TotalRatings: {occ[1]}, AverageRating: {occ[2]}")

Bài 6: Phân Tích Đánh Giá Theo Thời Gian

Mục tiêu:
Tính tổng số lượt đánh giá và điểm trung bình cho mỗi năm.

Output: Year - TotalRatings: xx, AverageRating: xx


In [ ]:
print("\n" + "="*80)
print("BÀI 6: PHÂN TÍCH ĐÁNH GIÁ THEO THỜI GIAN")
print("="*80)

# Extract year from timestamp and map to (Year, (Rating, 1))
year_ratings = all_ratings_rdd.map(
    lambda x: (datetime.fromtimestamp(int(x[3])).year, (x[2], 1))
)

# Reduce by year
year_aggregated = year_ratings.reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))

# Calculate average and format output
year_stats = year_aggregated.map(
    lambda x: (x[0], x[1][1], round(x[1][0] / x[1][1], 2))
).sortBy(lambda x: x[0])

print("\nYearly Ratings:")
for year in year_stats.collect():
    print(f"Year {year[0]} - TotalRatings: {year[1]}, AverageRating: {year[2]}")


In [7]:
# Stop Spark
spark.stop()